# 05 – Directly mapped types and final output

Reads the output of **04** (`nisjemodel-remote-sensing_norge_2026_25833.geo.parquet`)
and overrides the classification where directly mapped areas exist:

1. **NA-MC08 – Marint bunnsystem preget av oksygenmangel**
   from the MDir water type classification `NyTypologi2022`, types **1a, 2a, 3a**
   (`… med oksygenfattig bunnvann`, see `mnk.rules.WATER_TYPE_NAMES`).
   Only bottom with `nkm_lys = afotisk`.
2. **NA-MJ01 – Taretrålingsbunn**
   from NIVA kelp trawling areas 2020–2025. Applies to all bottom inside the areas.

The rules are defined in `mnk.rules.DIRECT_RULES`. Layers are applied in order; a later layer overrides an earlier one where they overlap.
Only affected polygons (spatial index) go through geometry operations.

Finally everything is split along the SSB 1 km grid (`rute_fkid`) and written as
`marint-naturkart_norge_2026_25833` (parquet + gpkg).

In [ ]:
import gc
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

IN_PARQUET = REPO / f"{mnk.utils.to_filename('nisjemodel-remote-sensing', 'norge', '2026', '25833')}.geo.parquet"
GRID_URL = "gs://niva-geodata/MarintNaturKart/aux/Basisdata_0000_Norge_25833_StatistiskRutenett1km.geo.parquet"

for rule in mnk.rules.DIRECT_RULES:
    print(rule["nin_kode_fkid"], mnk.rules.NIN_NAMES[rule["nin_kode_fkid"]], rule["url"])

## 1. Load data

In [ ]:
final = gpd.read_parquet(IN_PARQUET)
assert final.crs.to_epsg() == 25833
print(f"Input (04): {len(final):,} polygons")


def load_mask(rule):
    """Read the direct dataset of *rule* and return clean single polygons."""
    url = rule["url"]
    gdf = gpd.read_parquet(url) if url.endswith(".parquet") else gpd.read_file(url)
    if rule["column"] is not None:
        gdf = gdf.loc[gdf[rule["column"]].isin(rule["values"])]
    gdf = gdf[["geometry"]].to_crs(CRS)
    gdf = gdf.set_geometry(shapely.make_valid(gdf.geometry.to_numpy()), crs=CRS)
    gdf = gdf.explode(index_parts=False)
    gdf = gdf[gdf.geom_type == "Polygon"].reset_index(drop=True)
    print(f"{rule['nin_kode_fkid']}: {len(gdf):,} polygons ({gdf.area.sum() / 1e6:,.1f} km²)")
    return gdf

## 2. Generic override

`override` splits affected polygons into the part inside the mask (new type) and
the remainder (unchanged), same pattern as in 04.

In [ ]:
def override(final, mask, code, method, source, candidates=None):
    """Assign NiN type *code* to the parts of *final* inside *mask*."""
    if candidates is None:
        candidates = pd.Series(True, index=final.index)
    sub = final.loc[candidates]
    hit = sub.sindex.query(mask.geometry, predicate="intersects")
    hit_idx = sub.index[np.unique(hit[1])]

    affected = final.loc[hit_idx].copy()
    unaffected = final.drop(index=hit_idx)
    affected["_src"] = np.arange(len(affected))

    pairs = gpd.sjoin(affected, mask[["geometry"]], how="inner", predicate="intersects")
    mask_geom = mask.geometry.to_numpy()[pairs["index_right"].to_numpy()]
    inter = shapely.intersection(pairs.geometry.to_numpy(), mask_geom)
    inside = pairs.set_geometry(inter, crs=CRS).drop(columns="index_right")
    inside = inside.explode(index_parts=False)
    inside = inside[(inside.geom_type == "Polygon") & ~inside.geometry.is_empty]
    # Overlapping mask polygons give duplicates – dissolve per source polygon.
    inside = inside.dissolve(by="_src", as_index=False).explode(index_parts=False)
    inside = inside[inside.geom_type == "Polygon"]

    clip = inside.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
    src = affected.geometry.to_numpy()
    has = clip.notna().to_numpy()
    rest = src.copy()
    rest[has] = shapely.difference(src[has], clip[has].to_numpy())
    rest = affected.assign(geometry=rest).explode(index_parts=False)
    rest = rest[(rest.geom_type == "Polygon") & ~rest.geometry.is_empty]

    inside["nin_kode_fkid"] = code
    inside["NiN_name"] = mnk.rules.NIN_NAMES[code]
    inside["nin_int"] = pd.NA
    inside["nkm_prosess"] = mnk.attributes.NKM_PROSESS[code]
    inside["metode_klassifisering"] = method
    inside["datakilde"] = inside["datakilde"].map(
        lambda s: f"{s}|{source}" if isinstance(s, str) else source
    )

    cols = [c for c in unaffected.columns if not c.startswith("_")]
    out = gpd.GeoDataFrame(
        pd.concat([unaffected[cols], rest[cols], inside[cols]], ignore_index=True), crs=CRS
    )
    print(f"{code}: affected {len(affected):,} -> {len(inside):,} pieces, "
          f"{inside.area.sum() / 1e6:,.1f} km²")
    return out

## 3. Apply direct rules in order

Each rule in `mnk.rules.DIRECT_RULES` is burned in; a later rule overrides an
earlier one where they overlap.

In [ ]:
for rule in mnk.rules.DIRECT_RULES:
    mask = load_mask(rule)
    candidates = final["nkm_lys"].isin(rule["nkm_lys"]) if rule["nkm_lys"] else None
    final = override(
        final, mask, rule["nin_kode_fkid"],
        method=rule["metode_klassifisering"],
        source=rule["datakilde"],
        candidates=candidates,
    )
    del mask, candidates
    gc.collect()

print(f"Combined: {len(final):,} polygons")
print(final["nin_kode_fkid"].value_counts())

## 4. Split by SSB 1 km statistical grid

Polygons are split along the SSB 1 km grid cells and tagged with `rute_fkid`
(`ssbid1000M`). `nk_fid` is renewed so every split polygon gets a unique ID.

In [ ]:
grid = (
    gpd.read_parquet(GRID_URL, columns=["ssbid1000M", "geometry"])
    .rename(columns={"ssbid1000M": "rute_fkid"})
    .to_crs(CRS)
)
# Restrict the grid to the data extent.
minx, miny, maxx, maxy = final.total_bounds
grid = grid.cx[minx:maxx, miny:maxy]

# Split via sjoin + vectorised shapely intersection (faster than gpd.overlay).
pairs = gpd.sjoin(
    final, grid[["rute_fkid", "geometry"]], how="inner", predicate="intersects"
)
_ggeom = grid.geometry.to_numpy()[grid.index.get_indexer(pairs["index_right"])]
_inter = shapely.intersection(pairs.geometry.to_numpy(), _ggeom)
final = pairs.set_geometry(_inter, crs=CRS).drop(columns="index_right")
# Keep polygon parts only (explode GeometryCollections) to preserve area.
final = final.explode(index_parts=False)
final = final[final.geom_type == "Polygon"]
final = final[~final.geometry.is_empty & final.geometry.notna()].reset_index(drop=True)

# Renew unique nk_fid after splitting; place rute_fkid right after opphav_kode.
final["nk_fid"] = [f"NKM-{i:010d}" for i in range(1, len(final) + 1)]
cols = [c for c in final.columns if c != "geometry"]
cols.insert(cols.index("opphav_kode") + 1, cols.pop(cols.index("rute_fkid")))
final = final[cols + ["geometry"]]
print(f"After grid split: {len(final):,} polygons")

del grid, pairs, _ggeom, _inter
gc.collect()

## 5. Save

In [ ]:
fname = mnk.utils.to_filename("marint-naturkart", "norge", "2026", CRS.split(":")[1])
final.to_parquet(REPO / f"{fname}.geo.parquet", compression="snappy")
print(f"Saved {len(final):,} polygons -> {fname}.geo.parquet")

# Write GPKG in chunks to keep memory down at national scale.
gpkg_path = REPO / f"{fname}.gpkg"
gpkg_path.unlink(missing_ok=True)
CHUNK = 100_000
for start in range(0, len(final), CHUNK):
    final.iloc[start : start + CHUNK].to_file(
        gpkg_path, driver="GPKG", layer="nin", mode="w" if start == 0 else "a"
    )
    gc.collect()
print(f"Saved {len(final):,} polygons -> {fname}.gpkg")